# 使用 requests 進行網路爬蟲：臺灣證券交易所股票資料

本 Notebook 可直接在 **Google Colab** 執行，示範使用 `requests.get()` 與 `params=payload` 發送 GET 請求，取得臺灣證券交易所公開的股票資料。

## 學習目標
1. 使用 `requests` 發送 HTTP GET 請求。
2. 使用字典 `payload` 傳遞查詢參數。
3. 判斷 HTTP 狀態碼及 API 回應。
4. 將 JSON 資料轉成 Python 資料結構。
5. 使用迴圈篩選並顯示股票資料。

> 注意：證交所資料會依交易日及資料更新時間而變動。若當日尚未更新，可能取得前一交易日資料或暫時沒有資料。

## 範例一：取得全市場每日股票交易資料

資料來源為臺灣證券交易所 OpenAPI。此 API 不需要額外的 API Key，程式會取得回應中的股票清單，再篩選台積電（2330）。

In [ ]:
import requests

# 1. 定義查詢參數
# 此 API 不需要額外查詢條件，因此使用空字典
payload = {}

# 2. 設定 API 網址並發送 GET 請求
url = "https://openapi.twse.com.tw/v1/exchangeReport/STOCK_DAY_ALL"

try:
    r = requests.get(url, params=payload, timeout=30)

    # 3. 檢查 HTTP 回應狀態
    r.raise_for_status()

    # 4. 將 JSON 轉換成 Python 資料
    data = r.json()

    print("成功取得資料！")
    print("總共有", len(data), "筆股票資料")
    print("-" * 70)

    # 5. 篩選台積電（股票代號 2330）
    found = False

    for stock in data:
        if stock.get("Code") == "2330":
            found = True
            print("股票代號：", stock.get("Code"))
            print("股票名稱：", stock.get("Name"))
            print("交易日期：", stock.get("Date"))
            print("成交股數：", stock.get("TradeVolume"))
            print("成交金額：", stock.get("TradeValue"))
            print("開盤價：", stock.get("OpeningPrice"))
            print("最高價：", stock.get("HighestPrice"))
            print("最低價：", stock.get("LowestPrice"))
            print("收盤價：", stock.get("ClosingPrice"))
            break

    if not found:
        print("目前回傳資料中找不到台積電（2330）。")

except requests.exceptions.RequestException as e:
    print("網路請求發生錯誤：", e)
except ValueError:
    print("回傳內容不是有效的 JSON 格式。")


### 範例一程式說明

| 程式語法 | 說明 |
|---|---|
| `import requests` | 載入 requests 網路請求模組 |
| `payload = {}` | 建立查詢參數字典；本 API 不需要額外參數 |
| `requests.get(url, params=payload)` | 發送 GET 請求，並附加查詢參數 |
| `r.raise_for_status()` | 若 HTTP 回應為錯誤狀態，產生例外 |
| `r.json()` | 將 JSON 回應轉換成 Python 資料 |
| `for stock in data` | 逐筆讀取股票資料 |
| `stock.get("Code")` | 取得股票代號；使用 `get()` 可避免欄位不存在時直接發生 KeyError |

**觀念：** `params` 會將字典中的查詢參數編碼後附加到 URL。這是 GET 請求常見的參數傳遞方式。

## 範例二：指定股票代號及年月，查詢歷史成交資料

本例使用證交所歷史交易資料查詢端點，透過 `payload` 指定：
- `date`：查詢年月，格式為 `YYYYMMDD`，通常填該月份任一日期。
- `stockNo`：股票代號，例如台積電為 `2330`。
- `response`：指定回傳 JSON 格式。

**執行前可修改下方的日期與股票代號。** 請選擇已經結束的月份，以提高取得完整資料的機會。

In [ ]:
import requests

# 1. 定義查詢參數
payload = {
    "response": "json",
    "date": "20261001",   # 查詢年月：2026 年 10 月
    "stockNo": "2330"     # 股票代號：台積電
}

# 2. 設定網址並發送 GET 請求
url = "https://www.twse.com.tw/rwd/zh/afterTrading/STOCK_DAY"

try:
    r = requests.get(url, params=payload, timeout=30)
    r.raise_for_status()

    # 3. 將 JSON 轉換成 Python 字典
    result = r.json()

    # 4. 判斷證交所是否成功提供資料
    if result.get("stat") == "OK":
        print("查詢成功！")
        print("查詢標題：", result.get("title"))
        print("-" * 90)

        # 顯示欄位名稱
        print("資料欄位：")
        print(result.get("fields"))
        print("-" * 90)

        # 5. 逐筆顯示每日成交資料
        for row in result.get("data", []):
            print(row)

    else:
        print("查詢未成功，證交所回應：", result.get("stat"))
        print("請確認日期、股票代號，或稍後再試。")

except requests.exceptions.RequestException as e:
    print("網路請求發生錯誤：", e)
except ValueError:
    print("回傳內容不是有效的 JSON 格式。")


### 範例二程式說明

| 查詢參數 | 意義 | 範例 |
|---|---|---|
| `response` | 指定回傳格式 | `json` |
| `date` | 查詢日期（用來指定月份） | `20261001` |
| `stockNo` | 股票代號 | `2330` |

例如：

```python
payload = {
    "response": "json",
    "date": "20261001",
    "stockNo": "2330"
}

r = requests.get(url, params=payload)
```

requests 會自動將參數組合到網址後方，概念上類似：

```text
https://www.twse.com.tw/rwd/zh/afterTrading/STOCK_DAY?response=json&date=20261001&stockNo=2330
```

回傳資料中的 `fields` 是欄位名稱，`data` 則是每日成交資料。

## 課堂練習

請自行修改範例二，完成下列練習：

1. 將股票代號 `2330` 改成 `2317`（鴻海）。
2. 將查詢月份改成其他已結束的月份。
3. 觀察不同股票的成交資料筆數及收盤價。
4. 試著將每一筆資料整理成 pandas DataFrame。

## 小提醒
- HTTP 狀態碼 `200` 通常代表 HTTP 請求成功，但不一定代表 API 業務查詢成功；仍須檢查回傳 JSON 的 `stat`。
- 網站可能因維護、流量限制或服務調整而暫時無法回應。
- 請合理控制請求頻率，遵守網站使用規範。